In [ ]:
!pip install -q -U  huggingface_hub "bitsandbytes>=0.46.1" trl sentence-transformers


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
model_id = "gpt2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

def load_base_model():
    """Load a fresh, 4-bit quantized copy of the base model.
    Called once per adapter so adapters don't share/overwrite weights."""
    return AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
    )

model = load_base_model()


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

def make_lora_model(base_model):
    base_model = prepare_model_for_kbit_training(base_model)  #This prepares your 4-bit model for training with LoRA.
    peft_config = LoraConfig(
        r=16,                                 # Rank of update matrices
        lora_alpha=32,                        # Scaling factor
        target_modules=["c_attn", "c_proj"],  # Target layers for GPT-2
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",                #telling PEFT:This model is being used for causal language modeling.
    )
    peft_model = get_peft_model(base_model, peft_config)
    peft_model.print_trainable_parameters()  # Verifies base parameters are frozen
    return peft_model

peft_model = make_lora_model(model) #this model is gpt2 quantized
#using lora over quantized model,QLoRA

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

# NOTE: SFTTrainer calls formatting_func on ONE example at a time (not a batch),
# so it must return a single string, not a list. Returning [text] here was the
# bug behind: AttributeError: 'list' object has no attribute 'endswith'
def format_prompts(example):
    return f"User: {example['prompt']}\nAssistant: {example['completion']}"

def train_adapter(peft_model, train_data, output_dir):
    dataset = Dataset.from_list(train_data)     #converting into hf dataset(dictionary with splits as key)
    training_args = SFTConfig(
        output_dir=output_dir,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=10,
        max_steps=100,
        # fp16=True,
        fp16=False,  # avoid GradScaler/BFloat16 dtype clash with 4-bit quantized base
        bf16=False,
        save_strategy="no",
        max_length=512,
        completion_only_loss=False,
    )
    trainer = SFTTrainer(
        model=peft_model,
        train_dataset=dataset,
        formatting_func=format_prompts,
        args=training_args,
    )
    trainer.train()
    return trainer


In [ ]:
# 15 varied contract-law examples (was: 1) so the adapter has to learn the
# domain's vocabulary/style instead of memorizing a single sentence.
q1_train_data = [
    {"prompt": "What is the penalty for breach of Clause 4?",
     "completion": "A mandatory 30-day cure period applies before liquidated damages."},
    {"prompt": "What happens if a party fails to deliver goods on time?",
     "completion": "The non-breaching party may claim liquidated damages after a 15-day grace period."},
    {"prompt": "Can the contract be terminated early?",
     "completion": "Either party may terminate with 60 days written notice under Clause 9."},
    {"prompt": "What is the liability cap under this agreement?",
     "completion": "Total liability is capped at the fees paid in the preceding 12 months."},
    {"prompt": "Who is responsible for confidentiality breaches?",
     "completion": "The disclosing party must give written notice within 10 business days of discovering a breach."},
    {"prompt": "What remedies are available for non-payment?",
     "completion": "The vendor may suspend services after a 30-day payment default."},
    {"prompt": "How are disputes resolved under this contract?",
     "completion": "Disputes must first go through mediation before litigation is permitted."},
    {"prompt": "What is required to invoke force majeure?",
     "completion": "The affected party must notify the other party within 5 business days of the triggering event."},
    {"prompt": "Can the agreement be assigned to a third party?",
     "completion": "Assignment requires prior written consent from both parties."},
    {"prompt": "What indemnification obligations exist?",
     "completion": "Each party indemnifies the other against third-party claims arising from gross negligence."},
    {"prompt": "What is the notice period for amending the contract?",
     "completion": "Amendments require 30 days advance written notice to all parties."},
    {"prompt": "What happens if confidential information is leaked?",
     "completion": "The breaching party is liable for actual damages plus reasonable legal fees."},
    {"prompt": "Is there a warranty period for delivered services?",
     "completion": "Services carry a 90-day warranty against material defects."},
    {"prompt": "What governs this contract in case of conflict?",
     "completion": "The laws of the state of Delaware govern this agreement."},
    {"prompt": "What is the penalty for late delivery under Clause 7?",
     "completion": "A 2% penalty per week applies up to a maximum of 20% of contract value."},
]

train_adapter(peft_model, q1_train_data, "./results_q1")

# Save only the LoRA adapter weights (a few MB), NOT the full base model
peft_model.save_pretrained("./adapters/q1_contract_adapter")
tokenizer.save_pretrained("./adapters/q1_contract_adapter")


Applying formatting function to train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Step,Training Loss
10,3.601169
20,2.785523
30,2.485714
40,2.246236
50,2.055502
60,1.871982
70,1.753572
80,1.664696
90,1.607284
100,1.546067


('./adapters/q1_contract_adapter/tokenizer_config.json',
 './adapters/q1_contract_adapter/tokenizer.json')

In [ ]:
# 15 varied tariff/customs examples (was: 1)
q4_train_data = [
    {"prompt": "What is the tariff rate for cross-border electronics?",
     "completion": "A 12% customs duty applies to cross-border electronics shipments."},
    {"prompt": "What duty applies to imported textiles?",
     "completion": "Textile imports are subject to an 8% customs tariff."},
    {"prompt": "Are there exemptions for personal use items?",
     "completion": "Items valued under $200 are exempt from customs duty."},
    {"prompt": "What tariff applies to imported steel?",
     "completion": "Steel imports face a 25% tariff under current trade regulations."},
    {"prompt": "How are agricultural imports taxed at the border?",
     "completion": "Agricultural goods incur a 5% customs levy plus inspection fees."},
    {"prompt": "What happens if goods are misclassified under the wrong HS code?",
     "completion": "Misclassified goods are subject to a 15% penalty plus back duties."},
    {"prompt": "Do free trade agreement countries pay reduced tariffs?",
     "completion": "FTA partner countries qualify for a 0% preferential tariff rate."},
    {"prompt": "What is the duty on imported automobiles?",
     "completion": "Automobiles are taxed at a 10% import duty rate."},
    {"prompt": "Are pharmaceuticals subject to customs duty?",
     "completion": "Pharmaceutical imports are generally duty-free under most trade agreements."},
    {"prompt": "What tariff rate applies to luxury goods?",
     "completion": "Luxury goods incur a 20% customs surcharge."},
    {"prompt": "How much duty applies to raw materials?",
     "completion": "Raw materials typically carry a 3% customs duty."},
    {"prompt": "What is the penalty for undervaluing shipments?",
     "completion": "Undervaluation results in a 30% fine on the assessed value."},
    {"prompt": "Do e-commerce shipments face different tariffs?",
     "completion": "E-commerce shipments under $800 qualify for de minimis duty exemption."},
    {"prompt": "What tariff applies to imported furniture?",
     "completion": "Furniture imports are subject to a 9% customs duty."},
    {"prompt": "How are tariffs calculated for bulk commodity imports?",
     "completion": "Bulk commodities are assessed a 4% ad valorem duty based on shipment value."},
]

train_adapter(peft_model, q4_train_data, "./results_q4")

peft_model.save_pretrained("./adapters/q4_tariff_adapter")
tokenizer.save_pretrained("./adapters/q4_tariff_adapter")


Applying formatting function to train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Step,Training Loss
10,3.503659
20,2.715049
30,2.307178
40,2.003594
50,1.763795
60,1.661546
70,1.559675
80,1.462673
90,1.442336
100,1.376516


('./adapters/q4_tariff_adapter/tokenizer_config.json',
 './adapters/q4_tariff_adapter/tokenizer.json')

In [ ]:
from sentence_transformers import SentenceTransformer, util
from peft import PeftModel

# 1. Re-load Base Model & Attach All Saved Adapters
base_model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto"
)
model_with_adapters = PeftModel.from_pretrained(
    base_model, "./adapters/q1_contract_adapter", adapter_name="q1_contract"
)
model_with_adapters.load_adapter("./adapters/q4_tariff_adapter", adapter_name="q4_tariff")

# 2. Semantic Router (Small Embedding Model)
router_embedder = SentenceTransformer("all-MiniLM-L6-v2")
domain_descriptions = {
    "q1_contract": router_embedder.encode("corporate contract law breach liability cure period"),
    "q4_tariff": router_embedder.encode("cross border customs tariffs trade duties tax penalties"),
}

def generate_routed_response(query: str):
    # Route query by finding highest cosine similarity match
    query_emb = router_embedder.encode(query)
    best_adapter = max(
        domain_descriptions,
        key=lambda domain: util.cos_sim(query_emb, domain_descriptions[domain]),
    )

    # Activate the specific adapter dynamically
    model_with_adapters.set_adapter(best_adapter)

    # Run Inference
    inputs = tokenizer(query, return_tensors="pt").to(model_with_adapters.device)
    outputs = model_with_adapters.generate(
        **inputs,
        max_new_tokens=60,
        repetition_penalty=1.3,   # GPT-2 + heavy LoRA overfit tends to loop the memorized
        no_repeat_ngram_size=3,   # completion verbatim without these
        pad_token_id=tokenizer.eos_token_id,
    )
    return best_adapter, tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example Call:
adapter_used, response = generate_routed_response("What is the tariff percentage for cross-border electronics?")
print(f"[routed to: {adapter_used}]\n{response}")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[routed to: q4_tariff]
What is the tariff percentage for cross-border electronics?A 30% customs duty applies to imported goods.Cross border imports incur a 15%-30GASE tariffs plus GST per item under $100 value.FDI items are subject 1 Gase Duty compliance assessed at an inspection fee of 5%.Under current law cross country shipments receive preferential treatment with upcharge


In [ ]:
def generate_with_adapter(model, adapter_name, question, max_new_tokens=60):
    """Force a specific adapter on and generate -- used for isolation testing."""
    model.set_adapter(adapter_name)
    inputs = tokenizer(question, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)


def score_exact_recall(test_dataset, adapter_name, verbose=True):
    """MEMORIZATION test: questions lifted straight from the training set.
    A high score here just proves the adapter can recite what it was trained on."""
    correct = 0
    for item in test_dataset:
        decoded = generate_with_adapter(model_with_adapters, adapter_name, item["question"])
        hit = item["expected_keyword"].lower() in decoded.lower()
        correct += int(hit)
        if verbose:
            print(f"  [{adapter_name}] Q (in training set): {item['question']}")
            print(f"    -> {decoded!r}")
            print(f"    expected {item['expected_keyword']!r}: {'HIT' if hit else 'MISS'}\n")
    return (correct / len(test_dataset)) * 100


def score_generalization(test_dataset, adapter_name, verbose=True):
    """LEARNING test: questions the adapter has NEVER seen, phrased differently
    from anything in training. We don't expect a specific memorized fact back --
    we check whether the response uses the right domain's vocabulary/register at
    all, which only happens if the adapter learned the domain style rather than
    one string."""
    correct = 0
    for item in test_dataset:
        decoded = generate_with_adapter(model_with_adapters, adapter_name, item["question"]).lower()
        hit = any(kw.lower() in decoded for kw in item["domain_keywords"])
        correct += int(hit)
        if verbose:
            print(f"  [{adapter_name}] Q (held out, novel phrasing): {item['question']}")
            print(f"    -> {decoded!r}")
            print(f"    any of {item['domain_keywords']}: {'HIT' if hit else 'MISS'}\n")
    return (correct / len(test_dataset)) * 100


# --- MEMORIZATION test: near-verbatim questions from the 15 training examples ---
q1_recall_test = [
    {"question": "What is the penalty for breach of Clause 4?", "expected_keyword": "30-day"},
    {"question": "Can the contract be terminated early?", "expected_keyword": "60 days"},
]
q4_recall_test = [
    {"question": "What is the tariff rate for cross-border electronics?", "expected_keyword": "12%"},
    {"question": "What duty applies to imported textiles?", "expected_keyword": "8%"},
]

# --- GENERALIZATION test: novel scenarios, never seen verbatim, phrased differently ---
q1_generalization_test = [
    {"question": "If a vendor misses a delivery deadline, what recourse does the client have?",
     "domain_keywords": ["cure period", "liquidated damages", "notice", "breach", "grace period", "clause", "terminat"]},
    {"question": "Under what conditions can this agreement be brought to an early end?",
     "domain_keywords": ["terminat", "notice", "written", "clause", "days"]},
    {"question": "What protections exist if sensitive business information is disclosed improperly?",
     "domain_keywords": ["confidential", "indemnif", "damages", "breach", "notice", "disclos"]},
]
q4_generalization_test = [
    {"question": "What charges apply when bringing consumer electronics across the border?",
     "domain_keywords": ["duty", "tariff", "customs", "import", "%"]},
    {"question": "Is there a way to avoid paying import taxes on low-value packages?",
     "domain_keywords": ["exempt", "de minimis", "duty", "customs", "$"]},
    {"question": "How does misdeclaring a shipment's value affect the fees a company owes?",
     "domain_keywords": ["penalty", "fine", "duty", "valu", "%"]},
]

print("=" * 70)
print("MEMORIZATION CHECK (questions copied from the training set)")
print("=" * 70)
q1_recall_score = score_exact_recall(q1_recall_test, "q1_contract")
q4_recall_score = score_exact_recall(q4_recall_test, "q4_tariff")

print("=" * 70)
print("GENERALIZATION CHECK (novel questions, never seen, different phrasing)")
print("=" * 70)
q1_gen_score = score_generalization(q1_generalization_test, "q1_contract")
q4_gen_score = score_generalization(q4_generalization_test, "q4_tariff")

print("=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Q1 contract -- memorization (exact recall): {q1_recall_score:.1f}%   "
      f"generalization (novel, keyword-set): {q1_gen_score:.1f}%")
print(f"Q4 tariff   -- memorization (exact recall): {q4_recall_score:.1f}%   "
      f"generalization (novel, keyword-set): {q4_gen_score:.1f}%")

avg_recall = (q1_recall_score + q4_recall_score) / 2
avg_gen = (q1_gen_score + q4_gen_score) / 2
if avg_recall >= 80 and avg_gen < 50:
    print("\nVerdict: closer to MEMORIZATION -- strong recall on seen questions, "
          "weak transfer to novel ones.")
elif avg_gen >= 60:
    print("\nVerdict: closer to LEARNING -- the adapter carries the domain's "
          "vocabulary/register into questions it never saw.")
else:
    print("\nVerdict: mixed -- some generalization, but not strong. More/more-varied "
          "training data would likely help.")


MEMORIZATION CHECK (questions copied from the training set)
  [q1_contract] Q (in training set): What is the penalty for breach of Clause 4?
    -> "What is the penalty for breach of Clause 4?A mandatory 30-day cure period applies before liquidated damages.Offer valid until late termination or default under contract with third parties.Disclaimer: This release contains forward information that may be incorrect at time's end, including but not limited to confidential party communication and other material facts required to make a case"
    expected '30-day': HIT

  [q1_contract] Q (in training set): Can the contract be terminated early?
    -> "Can the contract be terminated early?The agreement requires 30 days written notice to all parties before liquidated damages are payable.Contract terminations must occur within 14 working weeks of each other's combined assets falling below $5 million per year.Deadlines for non-termination vary between contracts.Dispute Resolution Procedure:A third 